In [1]:
### simple rag pipline
import sys
from pathlib import Path

# Make both the repo root and src/ importable so the notebook's `src.*` /
# `data.*` imports resolve AND the bare intra-package imports inside the
# src modules (e.g. `from vector_store import VectorStore`) also resolve,
# regardless of where the notebook kernel is started.
repo_root = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
for p in (repo_root, repo_root / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from langchain_ollama import ChatOllama
from src.data_loader import process_all_txts, process_all_pdfs, process_all_urls

from data.url.url_list import url_list
from src.chunking import split_documents
from src.embedding import EmbeddingManager
from src.vector_store import VectorStore
from src.rag_retriever import RAGRetriever
from src.assistant import invoke, chat, generate_scan_command

# Process all PDFs, txts, and webpages
all_documents = (
    process_all_pdfs("../data")
    + process_all_txts("../data")
    + process_all_urls(url_list)
)
  
# chunking

chunks=split_documents(all_documents)

embedding_manager = EmbeddingManager()
vector_store = VectorStore()
vector_store.reset_collection()

### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vector_store.add_documents(chunks,embeddings)

rag_retriever = RAGRetriever(vector_store=vector_store, embedding_manager= embedding_manager)

llm = ChatOllama(model="gemma4:latest", temperature=0.1, max_completion_tokens=1024)


/Users/qmc/Neutrons/rag/.pixi/envs/default/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Found 1 PDF files to process

Processing: VERITAS_Spec_Sheet.pdf
  ✓ Loaded 1 pages

Total documents loaded: 1
Found 1 txt files to process

Processing: triple_axis_scan.txt
  ✓ Loaded 1 document(s)

Total documents loaded: 1
Found 7 URLs to process

Processing: https://neutrons.ornl.gov/veritas
  ✓ Extracted 8555 chars

Processing: https://neutrons.ornl.gov/veritas/team
  ✓ Extracted 5763 chars

Processing: https://neutrons.ornl.gov/veritas/users
  ✓ Extracted 8817 chars

Processing: https://neutrons.ornl.gov/veritas/capabilities
  ✓ Extracted 7767 chars

Processing: https://neutrons.ornl.gov/veritas/sample
  ✓ Extracted 6871 chars

Processing: https://neutrons.ornl.gov/veritas/publications
  ✓ Extracted 70067 chars

Processing: https://neutron.ornl.gov/hb1a/
  ✓ Extracted 1610 chars

Total documents loaded: 7
Split 9 documents into 146 chunks

Example chunk:
Content: Managed by UT-Battelle LLC  
for the US Department of Energy 
High Flux Isotope Reactor
 BEAMLINE 
HB-1A
 VERITAS  
Tr

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8852.55it/s]


Model loaded successfully. Embedding dimension: 384
Vector store initialied. Collection: documents
Existing documents in collection: 0
Resetting collection: documents
Vector store initialied. Collection: documents
Existing documents in collection: 0
generating embedding for 146 texts...


Batches: 100%|██████████| 5/5 [00:00<00:00, 11.33it/s]

Generated embeddings with shape : (146, 384)
Adding 146 documents to vector store....
  Added 129 documents (17 duplicates skipped).


In [ ]:
print(chat("How to get low bandwidth status display?", rag_retriever, llm,top_k=5))

Retrieving documents for query: 'How to get low bandwidth status display?'
Top K: 3, score threshold: 0.0
generating embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 23.33it/s]

Generated embeddings with shape : (1, 384)
Retrieved 3 documents (after filtering)
